# 4.3 - BERT and Encoder Models

**Module 4 - Model Architecture** | Netsetos GenAI Engineering

This notebook is a hands-on tour of BERT, the encoder-only Transformer that reads text in both directions at once. You'll watch BERT fill in masked words, peek at the vectors it produces, and then build a full movie-review sentiment analyzer four ways - from a 3-line pipeline to a model you fine-tune yourself on IMDB data. It closes by placing BERT inside the encoder/decoder/encoder-decoder family and the 2025 production landscape.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the Hugging Face and PyTorch stack this lesson runs on. Do this once per environment; skip it if these packages are already present.

In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers torch numpy datasets


**Explanation**

A one-line environment-prep cell - it installs the libraries, it doesn't run any model.

**How the code works - step by step**
- **`transformers`** - Hugging Face's model + `pipeline` library; provides BERT, the tokenizers, and the `Trainer`.
- **`torch`** - PyTorch, the tensor/back-prop engine BERT actually runs on.
- **`numpy`** - array math, used later to turn logits into accuracy.
- **`datasets`** - Hugging Face dataset loader, used to pull IMDB reviews in Stage C.
- **`-q`** - quiet flag; drop it if you want the full pip log.

**In one line:** install the four libraries the rest of the notebook imports.

**What you'll see:** (no output - environment prep)

## 1 - Watch BERT play fill-in-the-blank

How do you teach a model to understand language? Hide a word and make it guess. This is Masked Language Modeling (MLM), the exact game BERT was trained on, and here you run it live on a single sentence to see its top-5 guesses ranked by confidence.

In [ ]:
# =============================================
# MLM: The game BERT plays to learn language
# Hide a word, guess it from context.
# =============================================

# pip install transformers
from transformers import pipeline

# Load a pre-trained BERT model for fill-in-the-blank
unmasker = pipeline("fill-mask", model="bert-base-uncased")

# Give it a sentence with [MASK] where a word is hidden
sentence = "The cat [MASK] on the warm mat."

print(f"Sentence: {sentence}\n")
print("BERT's top 5 guesses for the hidden word:")

results = unmasker(sentence)
for i, r in enumerate(results):
    print(f"  {i+1}. '{r['token_str']}' ({r['score']:.1%} confident)")

# Output:
#   1. 'sat' (45.2% confident)     ← Correct!
#   2. 'lay' (18.7% confident)     ← Also makes sense
#   3. 'was' (12.3% confident)     ← Grammatically valid
#   4. 'slept' (8.1% confident)    ← Reasonable
#   5. 'landed' (3.4% confident)   ← Less likely but possible


**Explanation**

The whole cell is one `fill-mask` pipeline call - load a pre-trained BERT, hand it a sentence with a `[MASK]` hole, and read back what it thinks belongs there. It's a demonstration of a trained model, not any training.

**How the code works - step by step**
- **`pipeline("fill-mask", model="bert-base-uncased")`** - downloads BERT and wraps it so it predicts the masked token.
- **`sentence`** - a sentence with `[MASK]` standing in for the hidden word.
- **`unmasker(sentence)`** - returns a ranked list of candidate words, each with a `token_str` and a `score`.
- **the loop** - prints the top 5 guesses with their confidence as a percentage.

**In one line:** load BERT, mask one word, print its five best guesses.

**What you'll see:** The masked sentence, then a numbered top-5 list - 'sat' first at ~45%, followed by 'lay', 'was', 'slept', 'landed' - each with a confidence percentage. BERT used the words on both sides of the blank to pick 'sat'.

## 2 - Probe what BERT learned about the world

BERT was never handed a fact sheet - it learned geography, jobs, food, emotions and tech purely from playing fill-in-the-blank on billions of sentences. This cell fires five different probe sentences at it to show that knowledge fell out of the training game.

In [ ]:
# =============================================
# FUN: See what BERT has learned about the world
# Just from reading billions of sentences!
# =============================================

from transformers import pipeline

unmasker = pipeline("fill-mask", model="bert-base-uncased")

test_sentences = [
    # Does it know geography?
    "The capital of France is [MASK].",
    
    # Does it understand professions?
    "The doctor told the [MASK] to take rest.",
    
    # Does it understand food?
    "I ordered a plate of butter [MASK] for dinner.",
    
    # Does it understand emotions?
    "After winning the match, the team felt very [MASK].",
    
    # Does it understand code/tech?
    "Python is a popular programming [MASK].",
]

for sent in test_sentences:
    result = unmasker(sent)[0]  # Top 1 guess
    print(f"  {sent}")
    print(f"  → BERT guesses: '{result['token_str']}' ({result['score']:.0%})\n")

# BERT learned geography, professions, food, emotions, and tech
# ALL from just playing fill-in-the-blank billions of times!
# Nobody told it "Paris is the capital of France."
# It figured it out from the patterns in text.


**Explanation**

Same `fill-mask` pipeline as before, but run over a list of sentences that each test a different domain of knowledge. It's a quick survey, taking only the single top guess per sentence.

**How the code works - step by step**
- **`unmasker`** - the same `fill-mask` BERT pipeline.
- **`test_sentences`** - five masked prompts probing geography, professions, food, emotions and tech.
- **`unmasker(sent)[0]`** - runs each sentence and keeps only the #1 prediction (`[0]`).
- **the loop** - prints each prompt and BERT's single best fill with a rounded confidence.

**In one line:** ask five domain questions, print BERT's top answer for each.

**What you'll see:** Five sentences, each followed by BERT's single best guess and a percentage - e.g. it fills 'The capital of France is [MASK]' with 'paris'. The point: all of this knowledge came from fill-in-the-blank pre-training, not explicit teaching.

## 3 - Peek inside BERT: from tokens to vectors

Before using BERT you should see what it actually does to a sentence. This cell tokenizes 'I love this movie', shows the special `[CLS]`/`[SEP]` tokens BERT adds, then runs it through the model and inspects the 768-dim vector BERT produces for every token.

In [ ]:
# =============================================
# PEEK INSIDE BERT
# See how it processes a sentence into vectors
# =============================================

from transformers import BertTokenizer, BertModel
import torch

# Load BERT
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

# Tokenize a sentence
sentence = "I love this movie"
tokens = tokenizer(sentence, return_tensors="pt")

# See what tokens BERT created
token_ids = tokens["input_ids"][0]
token_words = tokenizer.convert_ids_to_tokens(token_ids)
print("Tokens BERT sees:")
for i, (tid, word) in enumerate(zip(token_ids, token_words)):
    print(f"  Position {i}: '{word}' (ID: {tid})")

# Output:
#   Position 0: '[CLS]'  (ID: 101)    ← Special start token
#   Position 1: 'i'      (ID: 1045)
#   Position 2: 'love'   (ID: 2293)
#   Position 3: 'this'   (ID: 2023)
#   Position 4: 'movie'  (ID: 3185)
#   Position 5: '[SEP]'  (ID: 102)    ← Special end token

# Pass through BERT
with torch.no_grad():
    outputs = model(**tokens)

# BERT gives us a vector for EACH token
all_vectors = outputs.last_hidden_state
print(f"\nOutput shape: {all_vectors.shape}")
# → (1, 6, 768)
# 1 sentence, 6 tokens, 768 numbers per token

# The [CLS] vector = a summary of the WHOLE sentence
cls_vector = all_vectors[0][0]
print(f"[CLS] vector (sentence summary): {cls_vector.shape}")
# → 768 numbers that capture the meaning of "I love this movie"
# We'll use this for sentiment analysis in Part 3!


**Explanation**

This drops from the high-level `pipeline` down to the raw `BertTokenizer` + `BertModel`, so you can watch the two stages directly: text becomes token IDs, then token IDs become vectors. The `[CLS]` vector it isolates at the end is the sentence summary you'll classify on later.

**How the code works - step by step**
- **`BertTokenizer` / `BertModel.from_pretrained(...)`** - load the tokenizer and the raw BERT encoder.
- **`tokenizer(sentence, return_tensors="pt")`** - turns the sentence into PyTorch tensors of token IDs.
- **`convert_ids_to_tokens` + loop** - prints each token, its position, and its ID, revealing the added `[CLS]` (101) and `[SEP]` (102).
- **`with torch.no_grad(): model(**tokens)`** - a forward pass with gradients off (inference only).
- **`outputs.last_hidden_state`** - the per-token vectors; its shape is printed as `(1, 6, 768)`.
- **`all_vectors[0][0]`** - grabs the `[CLS]` vector at position 0, the whole-sentence summary.

**In one line:** tokenize -> add [CLS]/[SEP] -> forward pass -> one 768-dim vector per token, with [CLS] as the sentence summary.

**What you'll see:** A list of six tokens with positions and IDs ([CLS], i, love, this, movie, [SEP]), then the output shape `(1, 6, 768)` (1 sentence, 6 tokens, 768 numbers each), then confirmation that the [CLS] vector is 768 numbers summarizing the sentence.

## 4 - Sentiment analysis in three lines

Instant gratification first. This is Stage A of the project: a sentiment classifier in three lines, because someone already fine-tuned a BERT model on movie reviews and Hugging Face lets you borrow it. You'll build your own version in Stage C.

In [ ]:
# =============================================
# STAGE A: Sentiment analysis in 3 lines
# This uses a BERT model already fine-tuned
# for sentiment analysis. No training needed!
# =============================================

from transformers import pipeline

# Load a pre-trained sentiment model (BERT inside)
sentiment = pipeline("sentiment-analysis")

# Analyze!
result = sentiment("I absolutely loved this movie! Best film of the year.")
print(result)
# → [{'label': 'POSITIVE', 'score': 0.9998}]

# That's it. 3 lines. BERT is doing all the heavy lifting inside.


**Explanation**

One `sentiment-analysis` pipeline call. It quietly downloads a default BERT-family model that's already fine-tuned for sentiment, so all you do is pass text and read a label. No training happens here.

**How the code works - step by step**
- **`pipeline("sentiment-analysis")`** - loads a pre-trained, pre-fine-tuned sentiment model.
- **`sentiment("...")`** - classifies one review string.
- **`print(result)`** - shows the returned label and confidence score.

**In one line:** load a ready-made sentiment model and classify one sentence.

**What you'll see:** A single-element list like `[{'label': 'POSITIVE', 'score': 0.9998}]` - BERT is ~99.98% confident the glowing review is positive.

## 5 - Batch a whole list of reviews

Stage B scales the toy to a list. Same model, but now you loop over seven reviews - including tricky mixed ones - and then tally how many came out positive versus negative.

In [ ]:
# =============================================
# STAGE B: Analyze a bunch of reviews at once
# =============================================

from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

reviews = [
    "Absolutely brilliant! The acting was superb.",
    "Worst movie I've ever seen. Complete waste of time.",
    "It was okay. Nothing special but not terrible either.",
    "The visuals were stunning but the story was weak.",
    "I walked out of the theater crying. So beautiful.",
    "Boring. I fell asleep halfway through.",
    "A masterpiece. Will watch it again and again!",
]

print("Movie Review Sentiment Analysis")
print("=" * 55)

for review in reviews:
    result = sentiment(review)[0]
    label = result["label"]
    score = result["score"]
    
    # Color-code the output
    emoji = "💚" if label == "POSITIVE" else "🔴"
    
    print(f"\n{emoji} {label} ({score:.0%})")
    print(f"   \"{review[:60]}...\"")

# =============================================
# Count positive vs negative
# =============================================
results = sentiment(reviews)
pos = sum(1 for r in results if r["label"] == "POSITIVE")
neg = len(results) - pos

print(f"\n{'=' * 55}")
print(f"Summary: {pos} positive, {neg} negative out of {len(reviews)} reviews")


**Explanation**

Reuses the `sentiment-analysis` pipeline, first one review at a time inside a loop for pretty per-review output, then once more over the whole list to count the split. It shows both the single-string and list-input calling styles of a pipeline.

**How the code works - step by step**
- **`sentiment = pipeline("sentiment-analysis")`** - same pre-fine-tuned model as Stage A.
- **`reviews`** - seven review strings spanning clearly positive, clearly negative, and mixed tone.
- **per-review loop** - calls `sentiment(review)[0]`, picks an emoji from the label, prints label + rounded score + a 60-char snippet.
- **`sentiment(reviews)`** - passes the whole list at once to get all results together.
- **`pos` / `neg` counts** - sums how many results are `POSITIVE`, derives the rest as negative, prints a summary line.

**In one line:** classify each review, print it, then tally positive vs negative.

**What you'll see:** A header, then one color-coded block per review (emoji + POSITIVE/NEGATIVE + percentage + a snippet), followed by a summary line like 'Summary: N positive, M negative out of 7 reviews'. Mixed reviews get forced into whichever side the model leans.

## 6 - Fine-tune your own BERT on IMDB

This is the heart of the project - Stage C, where you stop borrowing and train. You take DistilBERT, bolt a 2-class head onto it, and fine-tune it on 1000 IMDB reviews so it learns sentiment from real labeled data.

> **Needs a GPU** - trains for 3 epochs; runnable on CPU but slow.

In [ ]:
# =============================================
# STAGE C: Fine-tune BERT for sentiment analysis
# We'll use a small subset of IMDB reviews.
# =============================================

# pip install transformers datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
import numpy as np

# ── Step 1: Load the data ──
print("Loading IMDB movie reviews...")
dataset = load_dataset("imdb")

# Use a small subset for fast training (1000 train, 200 test)
small_train = dataset["train"].shuffle(seed=42).select(range(1000))
small_test  = dataset["test"].shuffle(seed=42).select(range(200))

# Peek at the data
print(f"\nExample review (first 100 chars):")
print(f"  Text:  '{small_train[0]['text'][:100]}...'")
print(f"  Label: {small_train[0]['label']} (0=negative, 1=positive)")

# ── Step 2: Prepare the data for BERT ──
print("\nTokenizing reviews for BERT...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,      # cut long reviews to 512 tokens
        padding="max_length",  # pad short reviews to same length
        max_length=256,        # keep it manageable
    )

train_data = small_train.map(tokenize, batched=True)
test_data  = small_test.map(tokenize, batched=True)

# ── Step 3: Load BERT with a classification head ──
print("Loading DistilBERT + adding classification head...")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,   # 2 classes: positive and negative
)

# ── Step 4: Set up training ──
training_args = TrainingArguments(
    output_dir="./my_sentiment_model",
    num_train_epochs=3,            # 3 passes through the data
    per_device_train_batch_size=16, # 16 reviews at a time
    per_device_eval_batch_size=16,
    eval_strategy="epoch",    # test after each pass
    logging_steps=50,
    save_strategy="no",             # don't save checkpoints (for speed)
)

# Accuracy calculator
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=-1)
    accuracy = (predictions == eval_pred.label_ids).mean()
    return {"accuracy": accuracy}

# ── Step 5: TRAIN! ──
print("\nTraining started!\n")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

trainer.train()

# ── Step 6: See the results! ──
results = trainer.evaluate()
print(f"\nFinal accuracy: {results['eval_accuracy']:.1%}")
print("(With just 1000 training examples! More data = better accuracy)")


**Explanation**

A complete fine-tuning loop wrapped in the Hugging Face `Trainer`. Read it as six stages: load data -> tokenize -> load model-with-head -> configure training -> train -> evaluate. Nothing is written to disk (checkpoints off) to keep it fast; the trained model lives in memory for Stage D.

**How the code works - step by step**
- **`load_dataset("imdb")` + `.shuffle().select(...)`** - pulls IMDB and takes a small 1000-train / 200-test subset for speed.
- **`tokenize()` + `.map(..., batched=True)`** - runs DistilBERT's tokenizer with truncation and padding to a fixed 256-length so every review is the same size.
- **`AutoModelForSequenceClassification.from_pretrained(..., num_labels=2)`** - loads pre-trained DistilBERT and attaches a fresh 2-output classification head.
- **`TrainingArguments`** - sets 3 epochs, batch size 16, per-epoch eval, no checkpoint saving.
- **`compute_metrics()`** - argmaxes the logits and compares to labels to report accuracy.
- **`Trainer(...).train()`** - runs the actual fine-tuning loop over the tokenized data.
- **`trainer.evaluate()`** - measures final accuracy on the held-out test set.

**In one line:** load a slice of IMDB -> tokenize -> add a 2-class head to DistilBERT -> train 3 epochs -> print accuracy.

**What you'll see:** Progress prints for each stage (loading, an example review + its 0/1 label, tokenizing, training), the Trainer's live loss/accuracy logs, and a final line like 'Final accuracy: 8x.x%' - strong accuracy from just 1000 examples.

## 7 - Deploy the model you just trained

Stage D closes the loop: wrap YOUR freshly fine-tuned model in a pipeline and run it on five brand-new reviews it has never seen. This is what using your own model in production looks like.

In [ ]:
# =============================================
# STAGE D: Use your fine-tuned model
# =============================================

from transformers import pipeline

# Load YOUR trained model (not someone else's!)
my_model = pipeline(
    "sentiment-analysis",
    model=model,          # the model we just trained
    tokenizer=tokenizer,
)

# Test with reviews the model has NEVER seen
test_reviews = [
    "The acting was phenomenal. I was on the edge of my seat!",
    "I want my 2 hours back. Terrible plot, terrible acting.",
    "Not bad, not great. A decent one-time watch.",
    "My kids loved it and honestly, so did I!",
    "The trailer was better than the actual movie.",
]

print("Your custom sentiment model says:\n")
for review in test_reviews:
    result = my_model(review)[0]
    label = "POSITIVE" if result["label"] == "LABEL_1" else "NEGATIVE"
    score = result["score"]
    emoji = "👍" if label == "POSITIVE" else "👎"
    
    print(f"  {emoji} {label} ({score:.0%})")
    print(f"     \"{review[:55]}...\"\n")


**Explanation**

Builds a `sentiment-analysis` pipeline around the in-memory `model` and `tokenizer` from Stage C instead of a downloaded one. The only twist is that a from-scratch head emits generic `LABEL_0`/`LABEL_1` names, so the code maps `LABEL_1` back to POSITIVE.

**How the code works - step by step**
- **`pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)`** - wraps the model and tokenizer you just trained, not a public one.
- **`test_reviews`** - five held-out reviews the model never trained on.
- **the loop** - calls `my_model(review)[0]`, translates `LABEL_1` -> POSITIVE / else NEGATIVE, picks a thumb emoji, prints label + rounded score + a 55-char snippet.

**In one line:** feed unseen reviews to your own model and print its verdicts.

**What you'll see:** Five blocks, each a thumbs-up/down emoji with POSITIVE/NEGATIVE, a confidence percentage, and a snippet of the review - proof your fine-tuned model generalizes to text it hasn't seen.

## 8 - Transfer learning: the two-stage paradigm

This cell names the big idea behind everything above: pre-train once on billions of words (Google did that, at ~$50K), then fine-tune on your task in minutes. It re-derives, in code, how the [CLS] vector becomes a classifier.

In [ ]:
# =============================================
# TRANSFER LEARNING — The two-stage paradigm
# =============================================

# Stage 1: PRE-TRAINING (Google did this for you)
#   - Trained on Wikipedia + BookCorpus (3.3B words)
#   - 64 TPUs × 4 days = ~$50,000+ compute
#   - Learned: grammar, facts, reasoning, world knowledge
#   - Output: bert-base-uncased (110M parameters)

# Stage 2: FINE-TUNING (you do this in 15 minutes)
#   - Take pre-trained BERT
#   - Add a tiny classification head (768 → 2 neurons)
#   - Train on YOUR data (even 200 samples works!)
#   - Result: 90%+ accuracy on sentiment, NER, QA, etc.

# This is like how ImageNet pre-training revolutionized
# computer vision — Sebastian Ruder called it "NLP's ImageNet moment"

# The [CLS] token: how a language model becomes a classifier
# Input:  [CLS] This movie was amazing [SEP]
# BERT:   Produces 768-dim vector for each token
# [CLS] vector → aggregates the whole sentence's meaning
# Classification head: [CLS] vector → Linear(768, 2) → positive/negative

from transformers import AutoModel, AutoTokenizer
import torch

model = AutoModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "This movie was absolutely fantastic!"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    cls_vector = outputs.last_hidden_state[:, 0, :]  # [CLS] = position 0

print(f"[CLS] vector shape: {cls_vector.shape}")  # [1, 768]
print(f"This 768-dim vector IS the sentence's meaning.")
print(f"Add Linear(768, 2) on top → instant sentiment classifier.")

**Explanation**

Mostly a richly-commented explainer of the pre-train-then-fine-tune split, with a small live snippet at the end. The runnable part re-extracts the [CLS] vector from raw BERT to show concretely what a classification head would sit on top of.

**How the code works - step by step**
- **comment block** - lays out Stage 1 (Google's expensive pre-training) vs Stage 2 (your cheap fine-tuning) and how `[CLS]` -> `Linear(768, 2)` makes a classifier.
- **`AutoModel.from_pretrained("bert-base-uncased")`** - loads raw BERT (no head).
- **`tokenizer(text, return_tensors="pt")`** - tokenizes one sentence.
- **`with torch.no_grad(): model(**inputs)`** - inference-only forward pass.
- **`outputs.last_hidden_state[:, 0, :]`** - slices out the position-0 `[CLS]` vector.
- **prints** - confirms the vector shape is `[1, 768]` and that adding `Linear(768, 2)` on top yields a sentiment classifier.

**In one line:** pre-train once (Google) -> extract the [CLS] vector -> add a linear layer -> classifier.

**What you'll see:** `[CLS] vector shape: torch.Size([1, 768])`, then two explanatory lines stating that the 768-dim vector is the sentence's meaning and that a `Linear(768, 2)` on top turns it into a sentiment classifier.

## 9 - The three Transformer architectures

One Transformer, split three ways. This cell instantiates an encoder (BERT), a decoder (GPT-2) and an encoder-decoder (T5) side by side so you can feel the difference between understanding, generating and transforming text.

In [ ]:
# =============================================
# THE THREE TRANSFORMER ARCHITECTURES
# =============================================
from transformers import pipeline

# 1. ENCODER-ONLY (BERT) — understands text
#    Attention: bidirectional (every token sees every other)
#    Tasks: classification, NER, extractive QA, embeddings
#    Models: BERT, RoBERTa, DeBERTa, ModernBERT, IndicBERT
classifier = pipeline("sentiment-analysis")
print(classifier("BERT understands text deeply"))

# 2. DECODER-ONLY (GPT) — generates text
#    Attention: causal (each token only sees past tokens)
#    Tasks: chat, code gen, creative writing, reasoning
#    Models: GPT-4, Claude, Gemini, LLaMA, DeepSeek
generator = pipeline("text-generation", model="gpt2")
print(generator("The future of AI is", max_length=20))

# 3. ENCODER-DECODER (T5) — transforms text
#    Attention: encoder=bidirectional, decoder=causal + cross-attention
#    Tasks: translation, summarization, seq-to-seq
#    Models: T5, BART, mBART, Flan-T5
summarizer = pipeline("summarization", model="t5-small")

# The "bank" example — why bidirectional matters:
# "I went to the bank to deposit money"
#   GPT encoding "bank": only sees "I went to the" → ambiguous!
#   BERT encoding "bank": sees "deposit money" too → financial bank!
# This is why BERT is better at UNDERSTANDING tasks.

# From Lesson 4.1: Word2Vec gave "bank" ONE vector (static)
# BERT gives "bank" DIFFERENT vectors depending on context
# → "river bank" ≠ "bank account" → contextual embeddings!


**Explanation**

A comparison cell that spins up three different `pipeline` types, one per architecture family, and then uses the classic 'bank' example in comments to explain why bidirectional attention makes encoders better at understanding. The T5 summarizer is loaded but not called.

**How the code works - step by step**
- **`pipeline("sentiment-analysis")`** - the encoder-only (BERT) branch; runs and prints a classification.
- **`pipeline("text-generation", model="gpt2")`** - the decoder-only (GPT) branch; generates a short continuation of 'The future of AI is'.
- **`pipeline("summarization", model="t5-small")`** - the encoder-decoder (T5) branch; loaded to represent seq-to-seq (not invoked).
- **the 'bank' comments** - contrast how causal GPT vs bidirectional BERT encode an ambiguous word, and tie back to Lesson 4.1's static-vs-contextual embeddings.

**In one line:** stand up BERT / GPT-2 / T5 together to contrast understand vs generate vs transform.

**What you'll see:** A sentiment result dict for the BERT line and a generated-text list for the GPT-2 line (a short, somewhat random continuation). The T5 summarizer only loads. The commented 'bank' example carries the conceptual payoff.

## 10 - BERT in 2025: variants, cost, production

A closing reference block. No model runs here - it prints a comparison table of the BERT family and lays out the cost argument that keeps encoders dominant in production, plus where they fit in RAG.

In [ ]:
# =============================================
# BERT IN 2025 — Variants, Cost, Production
# =============================================

# BERT Family Tree — what to use when
variants = {
    "BERT-base":     (110, 512,   "The original. Still works great."),
    "DistilBERT":    (66,  512,   "40% smaller, 60% faster, 95% accuracy"),
    "RoBERTa":       (125, 512,   "No NSP, more data → better than BERT"),
    "DeBERTa-v3":    (184, 512,   "Disentangled attention → best accuracy"),
    "ModernBERT":    (149, 8192,  "Dec 2024: 16x context, 2T tokens, code!"),
    "IndicBERT":     (33,  512,   "12 Indian languages (AI4Bharat/IIT-M)"),
}

print(f"{'Model':<16} {'Params':>7} {'Context':>8} Description")
print("-" * 70)
for name, (params, ctx, desc) in variants.items():
    print(f"  {name:<14} {params:>5}M {ctx:>7} {desc}")

# THE COST ARGUMENT — why BERT wins in production
print(f"""
COST: Classifying 1 million texts
  Fine-tuned BERT:     $2-5    (GPU compute)
  GPT-4o API:          $1,250  (token pricing)
  GPT-4o-mini API:     $67     (still 13x more expensive)

  BERT is 250-500x cheaper than GPT-4o for classification!
  And with 200 labeled samples, BERT matches GPT-4o accuracy.

ModernBERT (Dec 2024) — "What BERT would look like if built today":
  - 8,192 token context (vs BERT's 512)
  - Trained on 2 TRILLION tokens (vs BERT's 3.3B)
  - RoPE + FlashAttention-2 (from Lesson 4.4!)
  - First encoder model that understands CODE
  - Apache 2.0 license, 2x faster than DeBERTa

For Indian languages: IndicBERT covers Hindi, Telugu, Tamil, 
  Kannada, Malayalam, Bengali, Marathi, Gujarati + 4 more.
  Built by AI4Bharat (IIT Madras) on 9B tokens from IndicCorp.

BERT in RAG (Module 8):
  - Bi-encoder (Sentence-BERT): generates embeddings for retrieval
  - Cross-encoder: reranks top candidates for precision
  - Bi-encoder retrieves top 50 → Cross-encoder reranks to top 5 → LLM answers
""")


**Explanation**

A pure print/reference cell driven by a small dictionary. It's a lookup table and a cost/positioning briefing, not a computation - useful as a cheat-sheet for choosing an encoder variant.

**How the code works - step by step**
- **`variants` dict** - maps six models (BERT-base, DistilBERT, RoBERTa, DeBERTa-v3, ModernBERT, IndicBERT) to (params in M, context length, one-line description).
- **table loop** - prints a formatted `Model / Params / Context / Description` table from the dict.
- **cost block** - a triple-quoted string comparing fine-tuned BERT (~$2-5) vs GPT-4o (~$1,250) vs GPT-4o-mini (~$67) to classify 1M texts, plus ModernBERT, IndicBERT and RAG bi-encoder/cross-encoder notes.

**In one line:** print a BERT-family comparison table and the production cost/positioning case.

**What you'll see:** A formatted six-row table of models with parameter counts, context lengths and descriptions, followed by a multi-line briefing on classification cost (BERT is 250-500x cheaper than GPT-4o), ModernBERT's upgrades, IndicBERT's language coverage, and BERT's bi-encoder/cross-encoder roles in RAG. No model call - it's a reference summary.

Across these ten blocks you saw the one idea BERT is built on - masked-word prediction over bidirectional context - turn into a working classifier: fill-mask reveals what BERT knows, the [CLS] vector becomes a 768-dim sentence summary, and a tiny linear head on top of a pre-trained model gives you sentiment analysis for the price of fine-tuning rather than pre-training. The four-stage project (borrow a model, batch it, fine-tune your own, deploy it) is the transfer-learning loop you'll reuse for any classification task. Next up in Module 4 you move from understanding-models to the full Transformer internals (attention, RoPE, FlashAttention) that both BERT and its decoder cousins share, and later modules put these same encoders to work as retrievers and rerankers in RAG.